# L07 · Policy Gradients and REINFORCE

## Goal

- connect the log-derivative trick to code
- compute reward-to-go
- explain the role of a baseline

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L07:toy:42").hexdigest()
print(f"lesson=L07 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L07 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:94c087d9fa42e86dbbcd15d9ad9a476a7cc879c6a3e4e6b1ee39caf7c356b141 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: probability/gradients → **Policy Gradient and REINFORCE** → Actor-Critic/PPO

$$\nabla_\theta J(\theta)=\mathbb{E}\left[\nabla_\theta\log\pi_\theta(a_t\mid s_t)(G_t-b(s_t))\right]$$

The log-derivative trick estimates an expectation gradient by multiplying sampled-action log-probability by return. Reward-to-go assigns future rewards to earlier actions. An action-independent baseline reduces variance without changing the expected gradient.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** What is the loss-gradient sign for a chosen log-probability multiplied by positive advantage? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>Negative. Gradient descent increases log-probability only when the loss derivative is negative.</details>

In [2]:
from rl_study.algorithms.policy_gradient import reinforce_loss
from rl_study.algorithms.tabular import monte_carlo_returns
chosen_log_probs = torch.tensor([-0.7, -0.5, -0.2], requires_grad=True)
rewards = torch.tensor([0.0, 0.0, 1.0])
reward_to_go = monte_carlo_returns(rewards, gamma=0.9)
pg_loss = reinforce_loss(chosen_log_probs, reward_to_go, baseline=0.2)
pg_loss.backward()
print({"returns": reward_to_go.tolist(),
       "loss": round(float(pg_loss.detach()), 4),
       "logprob_gradient": chosen_log_probs.grad.tolist()})

{'returns': [0.809999942779541, 0.8999999761581421, 1.0], 'loss': 0.3123, 'logprob_gradient': [-0.20333331823349, -0.23333333432674408, -0.2666666805744171]}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Returns and baselines are detached with respect to policy parameters so credit values are not recreated through the actor. Actor-critic generalizes this with a learned baseline.

**Common trap:** Memorizing the sign verbally often fails when switching between maximize and minimize forms. Assert directly that positive advantage gives a negative log-probability gradient. Regression tests: `test_reinforce_sign`, `test_actor_advantage_detached`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert chosen_log_probs.grad[-1].item() < 0.0
assert reward_to_go[-1].item() == 1.0
print("checks=passed")

checks=passed


**Recall:** Why can an action-dependent baseline bias a policy-gradient estimate? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** All three reward-to-go values were positive and all chosen-log-probability gradients were negative. Larger advantages produced larger magnitudes.
- Executable checks: `test_reinforce_sign`, `test_actor_advantage_detached`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L08 learns the baseline as a critic, tunes bias/variance with GAE, and limits updates with PPO.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/classic.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`